# Classification Results — Post-Processing & Visualization

This notebook loads detection results from any audio classification tool, merges them, generates plots, and extracts audio clips for detections of interest.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read results and save outputs
2. **Loads and merges** all result files from a folder you choose
3. **Filters** results by confidence threshold and species
4. **Generates plots** — heatmaps, scatter plots, species richness charts
5. **Extracts audio segments** — short clips for each detection, sourced from Arbimon or your Drive
6. **Reviews extracted segments** — listen to each clip, view its spectrogram, and mark it as true or false

### Before you start:
- Your result files must be CSV or TXT files with one detection per row
- You need a **Google account** with Google Drive
- In case of extracting audio segments from an Arbimon project, you need an **Arbimon account** (at [arbimon.org](https://arbimon.org)) with access to this project

### How to run:
Run each cell **one at a time**, top to bottom. Click the ▶ button or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` has not been run yet. After running it shows a number like `[1]`. Red text means an error — read the message, it usually tells you exactly what to fix.

---
### Table of contents
- [Step 1 — Connect Google Drive & Install Software](#step-1)
- [Step 2 — Load & Merge Results](#step-2)
- [Step 3 — Filter Results](#step-3)
- [Step 4 — Generate Plots](#step-4)
- [Step 5 — Extract Audio Segments](#step-5)
- [Step 6 — Review Extracted Segments](#step-6)

---

Created by [biodiversica](https://biodiversica.xyz). For issues, questions, or feedback, open an issue on [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) or visit [biodiversica.xyz](https://biodiversica.xyz).

---
## Step 1 — Connect Google Drive & Install Software <a name="step-1"></a>

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

!pip install pandas matplotlib seaborn librosa soundfile -q

print('All packages installed successfully.')
print('(The Arbimon SDK is installed later, only if you use Arbimon as the audio source.)')

---
## Step 2 — Load & Merge Results <a name="step-2"></a>

Point the notebook to the folder on your Google Drive where your result files are stored,
**or** directly to a single merged result file (such as the `MODEL_NAME.merged.results.csv`
produced by the *Audio Classification from Embeddings* notebook).
When you point to a folder, it may have one subfolder per recording site — all `.txt` and `.csv`
files inside are found and merged automatically.

In [111]:
#@title 📁 Results folder { display-mode: "form" }
#@markdown Full path to the folder on your Google Drive that contains the result files,
#@markdown **or** the full path to a single merged result file (e.g. `..._merged.results.csv`).
#@markdown When you point to a folder, its subfolders (one per recording site) are searched automatically.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/classification_results"  #@param {type:"string"}

import os, glob as _g
if os.path.isfile(DRIVE_RESULTS_DIR):
    print(f"Merged file    : {DRIVE_RESULTS_DIR}")
    print("Result files   : 1 file (single merged file)")
elif os.path.isdir(DRIVE_RESULTS_DIR):
    n = len(_g.glob(f'{DRIVE_RESULTS_DIR}/**/*.txt', recursive=True) +
            _g.glob(f'{DRIVE_RESULTS_DIR}/**/*.csv', recursive=True))
    print(f"Folder found   : {DRIVE_RESULTS_DIR}")
    print(f"Result files   : {n} file(s) detected")
else:
    print(f"WARNING: Path not found: {DRIVE_RESULTS_DIR}")
    print("Check the path above and make sure Google Drive is connected (Step 1).")

In [112]:
#@title ⚙️ File format & column settings { display-mode: "form" }
#@markdown Configure how your result files are structured.
#@markdown Defaults match the **Arbimon AI Audio Analyzer** output.
#@markdown
#@markdown ---
#@markdown **File format** — file extension to look for in the results folder.
FILE_FORMAT = "txt"  #@param ["csv", "txt"]

#@markdown **Delimiter** — character that separates columns in each row.
DELIMITER = "comma (,)"  #@param ["comma (,)", "tab", "semicolon (;)"]

#@markdown ---
#@markdown ### Column names
#@markdown Enter the exact column header as it appears in your result files.
#@markdown Leave blank for optional columns that your files do not contain.

#@markdown **Species / label**
COL_LABEL = "label"  #@param {type:"string"}

#@markdown **Recording site**
COL_SITE = "site"  #@param {type:"string"}

#@markdown **Date** *(YYYY-MM-DD)*
COL_DATE = "date"  #@param {type:"string"}

#@markdown **Time** *(HH:MM:SS)*
COL_TIME = "time"  #@param {type:"string"}

#@markdown **Confidence score**
COL_CONFIDENCE = "confidence"  #@param {type:"string"}

#@markdown **Detection start time** *(seconds within the recording)*
COL_START_TIME = "start_time"  #@param {type:"string"}

#@markdown **Detection end time** *(seconds within the recording)*
COL_END_TIME = "end_time"  #@param {type:"string"}

#@markdown ---
#@markdown *The two columns below are only needed when extracting audio from Arbimon.*

#@markdown **Stream / recording ID** *(Arbimon only)*
COL_STREAM_ID = "stream_id"  #@param {type:"string"}

#@markdown **UTC offset** *(Arbimon only — e.g. UTC-3)*
COL_UTC_OFFSET = "utc_offset"  #@param {type:"string"}

print('Format settings saved. Run the next cell to load the files.')

In [113]:
#@title 📊 Load and merge all result files { display-mode: "form" }
#@markdown Reads every result file in your folder and combines them into one table.
#@markdown Files are cached in local Colab storage after the first run for faster re-runs.
#@markdown
#@markdown **Force reload from Drive** — re-scans your Drive folder and copies any new or changed files.
#@markdown Leave unchecked for fast re-runs using the local cache.
FORCE_RELOAD = False  #@param {type:"boolean"}

import os, glob, shutil, warnings
import pandas as pd
warnings.filterwarnings('ignore')

_delim_map = {"comma (,)": ",", "tab": "\t", "semicolon (;)": ";"}
_sep = _delim_map.get(DELIMITER, ",")

LOCAL_RESULTS_DIR = '/content/results_local'
os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

# DRIVE_RESULTS_DIR may be a folder of result files OR a single merged result file.
if os.path.isfile(DRIVE_RESULTS_DIR):
    _scan_root   = os.path.dirname(DRIVE_RESULTS_DIR)
    _single_file = DRIVE_RESULTS_DIR
else:
    _scan_root   = DRIVE_RESULTS_DIR
    _single_file = None

local_files = glob.glob(f'{LOCAL_RESULTS_DIR}/**/*.{FILE_FORMAT}', recursive=True)

if local_files and not FORCE_RELOAD:
    print(f'Using {len(local_files)} cached file(s) from local storage. '
          f'(Check "Force reload" above to re-scan Drive.)')
else:
    print('Scanning Drive folder...')
    if _single_file is not None:
        drive_files = [_single_file]
    else:
        drive_files = glob.glob(f'{DRIVE_RESULTS_DIR}/**/*.{FILE_FORMAT}', recursive=True)
    if not drive_files:
        raise FileNotFoundError(
            f'No .{FILE_FORMAT} files found in: {DRIVE_RESULTS_DIR}\n'
            'Check the folder path, the file format setting, and make sure Google Drive is connected.'
        )
    copied = 0
    local_files = []
    for src in drive_files:
        rel = os.path.relpath(src, _scan_root)
        dst = os.path.join(LOCAL_RESULTS_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if FORCE_RELOAD or not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
        local_files.append(dst)
    print(f'Drive files found : {len(drive_files)} — copied {copied} (rest already cached).')

print('Reading files...')
dfs = []
for f in local_files:
    try:
        tmp = pd.read_csv(f, sep=_sep)
        if not tmp.empty:
            dfs.append(tmp)
    except Exception as e:
        print(f'  WARNING: could not read {os.path.basename(f)}: {e}')

if not dfs:
    raise ValueError('All result files were empty or unreadable.')

df_raw = pd.concat(dfs, ignore_index=True)

# Build rename map: user column name → internal standard name
_required_cols = {
    COL_LABEL:      'Common name',
    COL_SITE:       'Point name',
    COL_DATE:       'Date',
    COL_TIME:       'Time',
    COL_CONFIDENCE: 'Confidence',
    COL_START_TIME: 'start_time',
    COL_END_TIME:   'end_time',
}
_optional_cols = {}
if COL_STREAM_ID.strip():  _optional_cols[COL_STREAM_ID]  = 'stream_id'
if COL_UTC_OFFSET.strip(): _optional_cols[COL_UTC_OFFSET] = 'utc_offset'

_rename = {}
for src, dst in _required_cols.items():
    if not src.strip():
        print(f'  WARNING: no column name set for "{dst}" — check the format settings.')
    elif src not in df_raw.columns:
        print(f'  WARNING: column "{src}" not found in files (expected for "{dst}").')
    else:
        _rename[src] = dst

for src, dst in _optional_cols.items():
    if src in df_raw.columns:
        _rename[src] = dst

df_raw = df_raw.rename(columns=_rename)

df_raw['Date']       = pd.to_datetime(df_raw['Date'], errors='coerce').dt.date
df_raw['Time']       = pd.to_datetime(df_raw['Time'], format='%H:%M:%S', errors='coerce').dt.time
df_raw['Confidence'] = pd.to_numeric(df_raw['Confidence'], errors='coerce')
if 'start_time' in df_raw.columns:
    df_raw['start_time'] = pd.to_numeric(df_raw['start_time'], errors='coerce')
if 'end_time' in df_raw.columns:
    df_raw['end_time'] = pd.to_numeric(df_raw['end_time'], errors='coerce')

print(f'Files loaded       : {len(local_files)}')
print(f'Total detections   : {len(df_raw)}')
print(f'Recording sites    : {sorted(df_raw["Point name"].dropna().unique().tolist())}')
print(f'Unique labels      : {df_raw["Common name"].nunique()}')
print(f'Date range         : {df_raw["Date"].min()} → {df_raw["Date"].max()}')
print(f'Confidence range   : {df_raw["Confidence"].min():.3f} → {df_raw["Confidence"].max():.3f}')

---
## Step 3 — Filter Results <a name="step-3"></a>

Set the minimum confidence threshold and choose which species to include or exclude.
Run **Apply filters** afterwards — it will show a summary of what remains.

In [198]:
#@title 🔬 Filter settings { display-mode: "form" }

#@markdown **Minimum confidence** — detections below this score are discarded.
MIN_CONFIDENCE = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown **Maximum confidence** — detections above this score are discarded.
#@markdown Set to **1.0** to keep all detections up to the maximum.
MAX_CONFIDENCE = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown **Include all species?** — if checked, every label in the results is used.
#@markdown Uncheck to list specific species in the field below.
ALL_SPECIES = True  #@param {type:"boolean"}

#@markdown **Species to include** — only used when "Include all species" is unchecked.
#@markdown Separate names with commas.  Example: `Robin, Blackbird, Chiffchaff`
SPECIES_INCLUDE = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Species to exclude** — always applied, even when "Include all species" is checked.
#@markdown Useful for removing background-noise or unwanted labels.
#@markdown Separate names with commas.  Example: `Background noise, Unknown`
SPECIES_EXCLUDE = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### Date / time range
#@markdown **Filter by date range?** — if checked, only detections within the date range below are kept.
FILTER_DATE_RANGE = False  #@param {type:"boolean"}

#@markdown **Start date** *(YYYY-MM-DD, inclusive)*
DATE_START = "2025-06-01"  #@param {type:"string"}

#@markdown **End date** *(YYYY-MM-DD, inclusive)*
DATE_END = "2025-12-30"  #@param {type:"string"}

#@markdown ---
#@markdown **Filter by time of day?** — if checked, only detections within the time window below are kept.
#@markdown Useful for focusing on dawn chorus, nocturnal activity, etc.
FILTER_TIME_RANGE = False  #@param {type:"boolean"}

#@markdown **Start time** *(HH:MM, 24-hour, inclusive)*
TIME_START = "00:00"  #@param {type:"string"}

#@markdown **End time** *(HH:MM, 24-hour, inclusive)*
TIME_END = "23:59"  #@param {type:"string"}

print('Settings saved. Run the next cell to apply them.')

In [199]:
#@title ✅ Apply filters { display-mode: "form" }
#@markdown Run this cell after changing any filter setting above.
#@markdown It shows a summary of detections remaining per species.

import datetime as _dt

def _split_list(s):
    return [x.strip() for x in s.split(',') if x.strip()]

df = df_raw.copy()
df = df[(df['Confidence'] >= MIN_CONFIDENCE) & (df['Confidence'] <= MAX_CONFIDENCE)]

# ── Date range filter ─────────────────────────────────────────────────────────
if FILTER_DATE_RANGE:
    if DATE_START.strip():
        try:
            _d_start = _dt.date.fromisoformat(DATE_START.strip())
            df = df[df['Date'] >= _d_start]
        except ValueError:
            print(f'WARNING: invalid DATE_START "{DATE_START}" — expected YYYY-MM-DD. Skipping start filter.')
    if DATE_END.strip():
        try:
            _d_end = _dt.date.fromisoformat(DATE_END.strip())
            df = df[df['Date'] <= _d_end]
        except ValueError:
            print(f'WARNING: invalid DATE_END "{DATE_END}" — expected YYYY-MM-DD. Skipping end filter.')

# ── Time of day filter ────────────────────────────────────────────────────────
if FILTER_TIME_RANGE:
    try:
        _t_start = _dt.time.fromisoformat(TIME_START.strip())
        _t_end   = _dt.time.fromisoformat(TIME_END.strip())
        if _t_start <= _t_end:
            df = df[df['Time'].apply(
                lambda t: pd.notnull(t) and _t_start <= t <= _t_end)]
        else:
            # Overnight window (e.g. 22:00 – 04:00)
            df = df[df['Time'].apply(
                lambda t: pd.notnull(t) and (t >= _t_start or t <= _t_end))]
    except ValueError:
        print(f'WARNING: invalid time format — expected HH:MM. Skipping time filter.')

# ── Species filters ───────────────────────────────────────────────────────────
exclude_list = _split_list(SPECIES_EXCLUDE)
if exclude_list:
    df = df[~df['Common name'].isin(exclude_list)]

if ALL_SPECIES:
    target_species = sorted(df['Common name'].dropna().unique().tolist())
else:
    target_species = _split_list(SPECIES_INCLUDE)
    if not target_species:
        print('WARNING: "Include all species" is off but no names were listed — including all.')
        target_species = sorted(df['Common name'].dropna().unique().tolist())
    else:
        df = df[df['Common name'].isin(target_species)]

df['Common name'] = pd.Categorical(df['Common name'], categories=target_species, ordered=True)

# ── Full species pool for "Include all labels" in plots ───────────────────────
# Applies only the species include/exclude rules — not confidence, date, or time.
# This lets plots show rows for species that had 0 detections in the filtered period.
_pool = df_raw.copy()
if exclude_list:
    _pool = _pool[~_pool['Common name'].isin(exclude_list)]
if not ALL_SPECIES and target_species:
    _pool = _pool[_pool['Common name'].isin(target_species)]
all_label_rows = sorted(_pool['Common name'].dropna().unique().tolist())

print(f'Confidence range     : {MIN_CONFIDENCE} – {MAX_CONFIDENCE}')
if FILTER_DATE_RANGE:
    print(f'Date range           : {DATE_START or "start"} → {DATE_END or "end"}')
if FILTER_TIME_RANGE:
    print(f'Time of day          : {TIME_START} – {TIME_END}')
print(f'Detections remaining : {len(df)}')
print(f'Species included     : {len(target_species)}')
for sp in target_species:
    n = int((df['Common name'] == sp).sum())
    print(f'  {sp}: {n} detection(s)')
if exclude_list:
    print(f'Species excluded     : {exclude_list}')

In [200]:
#@title 💾 Export filtered results to CSV { display-mode: "form" }
#@markdown Saves the filtered detections as CSV files to a folder on your Google Drive.
#@markdown Run **Apply filters** (cell above) before running this cell.
#@markdown
#@markdown **Export to CSV?** — uncheck to skip the export.
EXPORT_CSV = False  #@param {type:"boolean"}

#@markdown **Output folder** — full path to a folder on your Google Drive where the CSV files will be saved.
DRIVE_CSV_DIR = "/content/drive/MyDrive/arbimon_results_csv"  #@param {type:"string"}

#@markdown **One file per species** — each species gets its own CSV file named after the species.
#@markdown If unchecked, all detections are saved in a single file (`filtered_results.csv`).
CSV_ONE_FILE_PER_SPECIES = True  #@param {type:"boolean"}

#@markdown **Include all columns?** — if checked, every column from the original data is kept.
#@markdown Uncheck to save only the core columns (label, site, date, time).
CSV_ALL_COLUMNS = False  #@param {type:"boolean"}

if not EXPORT_CSV:
    print('Export disabled. Check "Export to CSV" above and re-run to export.')
else:
    import os
    os.makedirs(DRIVE_CSV_DIR, exist_ok=True)

    _core_cols = [c for c in ['Common name', 'Point name', 'Date', 'Time',
                               'start_time', 'end_time'] if c in df.columns]
    df_export = df if CSV_ALL_COLUMNS else df[_core_cols]
    df_export = df_export.rename(columns={'Common name': 'label', 'Point name': 'site'})

    if CSV_ONE_FILE_PER_SPECIES:
        saved_files = []
        for sp in target_species:
            sp_df = df_export[df_export['label'] == sp]
            if sp_df.empty:
                continue
            safe_name = str(sp).replace(' ', '_').replace('/', '-')
            out_path  = os.path.join(DRIVE_CSV_DIR, f'{safe_name}.csv')
            sp_df.to_csv(out_path, index=False)
            saved_files.append((sp, len(sp_df), out_path))
        print(f'Saved {len(saved_files)} CSV file(s) to: {DRIVE_CSV_DIR}')
        for sp, n, path in saved_files:
            print(f'  {os.path.basename(path)}  ({n} detection(s))')
    else:
        out_path = os.path.join(DRIVE_CSV_DIR, 'filtered_results.csv')
        df_export.to_csv(out_path, index=False)
        print(f'Saved {len(df_export)} detection(s) to: {out_path}')

---
## Step 4 — Generate Plots <a name="step-4"></a>

Choose which plots to generate, set an output folder, and run the last cell.
All plots are saved as high-resolution PNG files to your Google Drive.

| Plot | What it shows |
|------|---------------|
| **Heatmap — species × site** | Detection count per species at each recording site (all dates combined) |
| **Heatmap — species × site per day** | Same heatmap split into one panel per recording day |
| **Scatter — confidence vs time (all sites)** | All sites in one figure, one subplot per site |
| **Scatter — confidence vs time (per site)** | One separate figure for each recording site |
| **Heatmap — species × hour of day** | Which hour of day each species was most active |
| **Bar chart — species richness** | Total and site-exclusive species count per site |

> Run Steps 2 and 3 first so the data is loaded and filtered before generating plots.

In [201]:
#@title 📊 Choose plots { display-mode: "form" }
#@markdown Check each plot you want to generate.

PLOT_HEATMAP_ALL = True  #@param {type:"boolean"}
#@markdown **Heatmap — detection count per species per site (all dates)**

PLOT_HEATMAP_PER_DAY = False  #@param {type:"boolean"}
#@markdown **Heatmap — same as above, one panel per recording day**

PLOT_SCATTER_CONFIDENCE = False  #@param {type:"boolean"}
#@markdown **Scatter — confidence vs time of day (all sites in one figure)**

PLOT_MULTIPLE_SCATTER = False  #@param {type:"boolean"}
#@markdown **Scatter — confidence vs time of day (one figure per site)**

PLOT_HEATMAP_SPECIES_HOUR = True  #@param {type:"boolean"}
#@markdown **Heatmap — detections per species per hour of day**

PLOT_RICHNESS = False  #@param {type:"boolean"}
#@markdown **Bar chart — species richness per site (unique vs total)**

#@markdown ---
#@markdown **Y-axis limit for the species richness chart**
MAX_SPECIES_AXIS = 20  #@param {type:"integer"}

#@markdown ---
#@markdown **Include all labels?** — if checked, every species from the filter step appears
#@markdown as a row in the heatmaps, even when it had 0 detections in the selected period.
#@markdown Uncheck to show only species that have at least one detection.
INCLUDE_ALL_LABELS = True  #@param {type:"boolean"}

print('Plot selection saved. Set the output folder and run plots in the next cell.')

In [202]:
#@title 🎨 Output folder and run plots { display-mode: "form" }
#@markdown All selected plots are saved as PNG files to this folder on your Google Drive.
DRIVE_PLOTS_DIR = "/content/drive/MyDrive/arbimon_plots"  #@param {type:"string"}

import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm

os.makedirs(DRIVE_PLOTS_DIR, exist_ok=True)

def _time_float(t):
    return t.hour + t.minute / 60.0 + t.second / 3600.0 if pd.notnull(t) else None

# Species row order: full pool (ignores confidence/date/time) or only those with detections
_label_rows = all_label_rows if INCLUDE_ALL_LABELS else sorted(df['Common name'].dropna().unique().tolist())

generated = []

# ── Heatmap: species × site ───────────────────────────────────────────────────
if PLOT_HEATMAP_ALL:
    print('Generating: heatmap — species × site...')
    cnt = df.groupby(['Common name', 'Point name']).size().reset_index(name='Count')
    hm  = (cnt.pivot(index='Common name', columns='Point name', values='Count')
              .reindex(index=_label_rows, columns=sorted(df['Point name'].dropna().unique()))
              .fillna(0))
    plt.figure(figsize=(12, max(4, len(hm) * 0.35)))
    sns.heatmap(hm, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.5, cbar=False,
                norm=PowerNorm(gamma=0.25, vmin=0, vmax=max(hm.values.max(), 1)))
    plt.title(f'Detecções por espécie e ponto  (Confiança ≥ {MIN_CONFIDENCE})', fontsize=13, fontweight='bold')
    plt.xlabel('Ponto'); plt.ylabel('Espécie')
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout()
    out = f'{DRIVE_PLOTS_DIR}/heatmap_species_vs_sites.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Saved: heatmap_species_vs_sites.png')

# ── Heatmap: species × site per day ──────────────────────────────────────────
if PLOT_HEATMAP_PER_DAY:
    print('Generating: heatmap — per day...')
    unique_dates = sorted(df['Date'].dropna().unique())
    if not unique_dates:
        print('  No date information found — skipping.')
    else:
        cnt  = df.groupby(['Date', 'Common name', 'Point name']).size().reset_index(name='Count')
        pts  = sorted(df['Point name'].unique())
        n    = len(unique_dates)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, max(4, 0.35 * len(_label_rows))))
        if n == 1: axes = [axes]
        for i, (ax, d) in enumerate(zip(axes, unique_dates)):
            hm = (cnt[cnt['Date'] == d]
                  .pivot(index='Common name', columns='Point name', values='Count')
                  .reindex(index=_label_rows, columns=pts).fillna(0))
            sns.heatmap(hm, ax=ax, annot=True, fmt='.0f', cmap='YlGnBu',
                        linewidths=0.5, linecolor='black', cbar=False, vmin=0,
                        norm=PowerNorm(gamma=0.25, vmin=0, vmax=max(hm.values.max(), 1)))
            ax.set_title(str(d), fontsize=11, fontweight='bold')
            ax.set_xlabel('Ponto')
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
            if i == 0:
                ax.set_ylabel('Espécie'); ax.set_yticklabels(_label_rows, rotation=0)
            else:
                ax.set_ylabel(''); ax.set_yticklabels([])
        plt.tight_layout(); plt.subplots_adjust(wspace=0.025)
        out = f'{DRIVE_PLOTS_DIR}/heatmap_species_vs_sites_per_day.png'
        plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
        print('  Saved: heatmap_species_vs_sites_per_day.png')

# ── Scatter setup (shared between both scatter plots) ─────────────────────────
if PLOT_SCATTER_CONFIDENCE or PLOT_MULTIPLE_SCATTER:
    _sc = df.copy()
    _sc['Date']       = pd.to_datetime(_sc['Date'], errors='coerce')
    _sc['time_float'] = _sc['Time'].apply(_time_float)
    palette   = sns.color_palette('tab10', max(len(target_species), 1))
    color_map = {sp: palette[i % len(palette)] for i, sp in enumerate(target_species)}
    leg_handles = [
        plt.Line2D([], [], color=color_map[sp], marker='o', linestyle='', markersize=6, label=sp)
        for sp in target_species
    ]

# ── Scatter: all sites in one figure ─────────────────────────────────────────
if PLOT_SCATTER_CONFIDENCE:
    print('Generating: scatter — all sites in one figure...')
    pts   = sorted(_sc['Point name'].unique())
    ncols = 2
    nrows = int(np.ceil(len(pts) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax in axes[len(pts):]: ax.set_visible(False)
    for ax, p in zip(axes, pts):
        sub = _sc[_sc['Point name'] == p]
        for sp in sub['Common name'].unique():
            s = sub[sub['Common name'] == sp]
            ax.scatter(s['time_float'], s['Confidence'],
                       color=color_map.get(str(sp), 'gray'), marker='o', s=40, alpha=0.8)
        ax.set_xticks(np.arange(0, 25, 1))
        ax.grid(True, linestyle='--', color='gray', alpha=0.4); ax.set_axisbelow(True)
        ax.set_title(str(p), fontsize=10, fontweight='bold')
        ax.set_xlabel('Horário'); ax.set_ylabel('Confiança')
        ax.set_xlim(0, 24); ax.set_ylim(MIN_CONFIDENCE, 1.0)
    fig.legend(leg_handles, target_species, loc='upper center', ncol=4,
               bbox_to_anchor=(0.5, 1.0), fontsize=9, frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.9])
    out = f'{DRIVE_PLOTS_DIR}/scatter_confidence_all_sites.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Saved: scatter_confidence_all_sites.png')

# ── Scatter: one figure per site ─────────────────────────────────────────────
if PLOT_MULTIPLE_SCATTER:
    print('Generating: scatter — one figure per site...')
    count_before = len(generated)
    for p in sorted(_sc['Point name'].unique()):
        sub = _sc[_sc['Point name'] == p]
        if sub.empty: continue
        fig, ax = plt.subplots(figsize=(10, 5))
        for sp in sub['Common name'].unique():
            s = sub[sub['Common name'] == sp]
            ax.scatter(s['time_float'], s['Confidence'],
                       color=color_map.get(str(sp), 'gray'), marker='o', s=40, alpha=0.8)
        ax.set_xticks(np.arange(0, 25, 1))
        ax.grid(True, linestyle='--', color='gray', alpha=0.4); ax.set_axisbelow(True)
        ax.set_title(str(p), fontsize=11, fontweight='bold')
        ax.set_xlabel('Horário'); ax.set_ylabel('Confiança')
        ax.set_xlim(0, 24); ax.set_ylim(MIN_CONFIDENCE, 1.0)
        fig.legend(leg_handles, target_species, loc='upper center', ncol=4,
                   bbox_to_anchor=(0.5, 1.02), fontsize=9, frameon=False)
        plt.tight_layout(rect=[0, 0, 1, 0.9])
        safe_p = str(p).replace(' ', '_').replace('/', '-')
        out = f'{DRIVE_PLOTS_DIR}/scatter_confidence_{safe_p}.png'
        plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print(f'  Saved {len(generated) - count_before} file(s).')

# ── Heatmap: species × hour ───────────────────────────────────────────────────
if PLOT_HEATMAP_SPECIES_HOUR:
    print('Generating: heatmap — species × hour of day...')
    _h = df.copy()
    _h['Hour'] = _h['Time'].apply(lambda t: t.hour if pd.notnull(t) else None)
    cnt = _h.groupby(['Common name', 'Hour']).size().reset_index(name='Count')
    hm  = (cnt.pivot(index='Common name', columns='Hour', values='Count')
              .reindex(index=_label_rows, columns=range(24)).fillna(0))
    plt.figure(figsize=(16, max(4, len(hm) * 0.35)))
    sns.heatmap(hm, annot=True, fmt='.0f', cmap='YlGnBu',
                linewidths=0.4, linecolor='black', cbar=False,
                norm=PowerNorm(gamma=0.3, vmin=0, vmax=max(hm.values.max(), 1)))
    plt.title(f'Detecções por espécie por hora  (Confiança ≥ {MIN_CONFIDENCE})', fontsize=13, fontweight='bold')
    plt.xlabel('Hora do dia'); plt.ylabel('Espécie'); plt.xticks(rotation=0)
    plt.tight_layout()
    out = f'{DRIVE_PLOTS_DIR}/heatmap_species_vs_hour.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Saved: heatmap_species_vs_hour.png')

# ── Bar chart: species richness ───────────────────────────────────────────────
if PLOT_RICHNESS:
    print('Generating: species richness bar chart...')
    pts   = sorted(df['Point name'].unique())
    total = df.groupby('Point name')['Common name'].nunique().reindex(pts)
    unique_counts = [
        len(set(df[df['Point name'] == p]['Common name']) -
            set(df[df['Point name'] != p]['Common name']))
        for p in pts
    ]
    pdfm = pd.DataFrame({'Point name': pts, 'Total': total.values, 'Únicas': unique_counts})
    pdfm = pdfm.melt(id_vars='Point name', value_vars=['Únicas', 'Total'],
                     var_name='Categoria', value_name='Contagem')
    plt.figure(figsize=(max(8, len(pts) * 1.2), 6))
    sns.barplot(data=pdfm, x='Point name', y='Contagem', hue='Categoria',
                palette=['#66c2a5', '#fc8d62'])
    plt.title(f'Espécies únicas vs total por ponto  (Confiança ≥ {MIN_CONFIDENCE})', fontsize=13, fontweight='bold')
    plt.xlabel('Ponto'); plt.ylabel('Número de espécies')
    plt.xticks(rotation=45, ha='right'); plt.ylim(0, MAX_SPECIES_AXIS); plt.legend(title=None)
    ax = plt.gca()
    for bar in ax.patches:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.2, f'{int(h)}',
                    ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    out = f'{DRIVE_PLOTS_DIR}/barplot_species_richness.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Saved: barplot_species_richness.png')

print()
print(f'Done! {len(generated)} plot(s) saved to: {DRIVE_PLOTS_DIR}')

---
## Step 5 — Extract Audio Segments <a name="step-5"></a>

This step cuts a short audio clip for each detection in your filtered results and saves it to your Google Drive.

**How it works:**
- Each detection stores the recording site, date, local time, and the exact position (in seconds) within the audio file.
- The notebook converts the local timestamps back to UTC, then downloads the recording from Arbimon or finds it in your Drive folder.
- The relevant segment is cut out and saved as a `.wav` file, organised into one subfolder per site.

**Before running:**
1. Complete Steps 1–3 so results are loaded and filtered.
2. Fill in the settings cells below, then run them top to bottom.

> **Tip:** Use the species filter in Step 3 to narrow down which detections to extract before running this step.

In [192]:
#@title ⚙️ Extraction settings { display-mode: "form" }

#@markdown **Audio source** — where are the original recordings stored?
AUDIO_SOURCE = "google_drive"  #@param ["arbimon", "google_drive"]

#@markdown ---
#@markdown ### If audio source = Google Drive
#@markdown Full path to the folder containing audio files on your Drive.
#@markdown Subfolders matching the site names are searched automatically.
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown ---
#@markdown **Output folder** — where extracted clips will be saved on your Drive.
#@markdown Clips are organised into one subfolder per label inside this folder.
DRIVE_SEGMENTS_DIR = "/content/drive/MyDrive/segments"  #@param {type:"string"}

#@markdown ---
#@markdown **Species to extract** — leave blank to extract all species in the filtered results.
#@markdown Separate names with commas.  Example: `Robin, Blackbird`
EXTRACT_SPECIES = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Maximum segments per label** — limits how many clips are extracted for each species.
#@markdown Set to **0** to extract all detections (no limit).
MAX_SEGMENTS_PER_LABEL = 5  #@param {type:"integer"}

#@markdown **Random seed** — controls which detections are selected when the limit above is applied.
#@markdown Use any positive integer for reproducible results, or **-1** for a different random draw each time.
RANDOM_SEED = -1  #@param {type:"integer"}

#@markdown ---
#@markdown **Padding (seconds)** — extra audio added before and after each detection window.
SEGMENT_PADDING_S = 0.0  #@param {type:"number"}

#@markdown **Output sample rate (Hz)** — rate for the saved WAV files.
#@markdown Use the same rate as your model (e.g. 48000 for BirdNET, 32000 for Google Perch).
EXTRACT_SR = 48000  #@param {type:"integer"}

import os
os.makedirs(DRIVE_SEGMENTS_DIR, exist_ok=True)
print(f'Audio source          : {AUDIO_SOURCE}')
print(f'Output folder         : {DRIVE_SEGMENTS_DIR}')
print(f'Max segments / label  : {"all" if MAX_SEGMENTS_PER_LABEL == 0 else MAX_SEGMENTS_PER_LABEL}')
print(f'Random seed           : {"off (random)" if RANDOM_SEED < 0 else RANDOM_SEED}')
print(f'Padding               : {SEGMENT_PADDING_S} s')
print(f'Sample rate           : {EXTRACT_SR} Hz')
if EXTRACT_SPECIES.strip():
    print(f'Species filter        : {[x.strip() for x in EXTRACT_SPECIES.split(",") if x.strip()]}')
else:
    print('Species filter        : all (from filtered results in Step 3)')

In [193]:
#@title 🔑 Log in to Arbimon (skip if using Google Drive) { display-mode: "form" }
#@markdown Run this cell only if your audio source is **Arbimon**.
#@markdown If you ran the Analyzer notebook in this session, you are already logged in — you can skip this.
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

if AUDIO_SOURCE == 'arbimon':
    # Install the Arbimon SDK here (not at the top) so it is only downloaded
    # when you actually use Arbimon as the audio source.
    !wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl -O /tmp/rfcx-0.3.1-py3-none-any.whl
    !pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
    import rfcx
    client_extract = rfcx.Client()
    if os.path.exists(CREDENTIALS_PATH):
        print('Credentials file found — logging in automatically...')
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    else:
        print('No saved credentials found.')
        print('A URL will appear below — open it in your browser and log in with your Arbimon account.')
        print('-' * 60)
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('\nLogged in to Arbimon successfully.')
else:
    print('Audio source is Google Drive — Arbimon login not needed. You can skip this cell.')

In [194]:
#@title ⬇️ Find or download recordings { display-mode: "form" }
#@markdown Identifies which original recordings contain the selected detections,
#@markdown then downloads them from Arbimon or locates them in your Drive folder.

import re, glob, datetime, unicodedata
import pandas as pd

# Build extraction subset
df_ext = df.copy()
if EXTRACT_SPECIES.strip():
    _ext_sp = [x.strip() for x in EXTRACT_SPECIES.split(',') if x.strip()]
    df_ext  = df_ext[df_ext['Common name'].isin(_ext_sp)]

if df_ext.empty:
    raise ValueError(
        'No detections match the extraction criteria.\n'
        'Check your species list and the filters applied in Step 3.'
    )

# ── Random sampling per label ─────────────────────────────────────────────────
if MAX_SEGMENTS_PER_LABEL > 0:
    _seed = RANDOM_SEED if RANDOM_SEED >= 0 else None
    df_ext = (
        df_ext
        .groupby('Common name', group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), MAX_SEGMENTS_PER_LABEL), random_state=_seed))
    )
    df_ext = df_ext.sort_index()
    print(f'Sampling applied      : up to {MAX_SEGMENTS_PER_LABEL} per label'
          f'  (seed={_seed if _seed is not None else "random"})')
    for sp, grp in df_ext.groupby('Common name'):
        print(f'  {sp}: {len(grp)} detection(s) selected')
else:
    print('Sampling              : disabled (extracting all detections)')

print(f'\nDetections to extract    : {len(df_ext)}')

def _utc_hours(utc_str):
    try:
        return int(str(utc_str).replace('UTC', '').strip() or 0)
    except ValueError:
        return 0

def _rec_datetime(row):
    # Returns the datetime of the recording start that contains this detection.
    # Used only to locate the audio file — start_time/end_time are offsets from
    # the start of that file (0 = beginning of the recording).
    # For Arbimon: converts local time to UTC using utc_offset.
    # For Google Drive: keeps local time (filenames use local time).
    try:
        local_dt = datetime.datetime.combine(row['Date'], row['Time'])
        if AUDIO_SOURCE == 'arbimon':
            local_dt = local_dt - datetime.timedelta(hours=_utc_hours(row.get('utc_offset', 'UTC+0')))
        return local_dt
    except Exception:
        return None

df_ext = df_ext.copy()
df_ext['_rec_dt'] = df_ext.apply(_rec_datetime, axis=1)

if AUDIO_SOURCE == 'arbimon':
    unique_recs = df_ext[['stream_id', '_rec_dt']].dropna().drop_duplicates()
else:
    # Keep the site: two sites often record at the same scheduled time, so the
    # recording is only uniquely identified by (site, datetime).
    unique_recs = df_ext[['Point name', '_rec_dt']].dropna().drop_duplicates()

print(f'Unique recordings needed : {len(unique_recs)}')

AUDIO_TEMP_DIR = '/content/audio_extract'
os.makedirs(AUDIO_TEMP_DIR, exist_ok=True)

audio_file_map = {}  # key → local audio file path

# ── Arbimon: one download call per needed recording ──────────────────────────
if AUDIO_SOURCE == 'arbimon':
    dest = AUDIO_TEMP_DIR
    os.makedirs(dest, exist_ok=True)

    def _arbimon_file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        return None

    for _, rec in unique_recs.iterrows():
        sid = str(rec['stream_id'])
        t0  = rec['_rec_dt']
        key = (sid, t0)

        # Check if already downloaded in a previous iteration
        existing = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                    glob.glob(f'{dest}/**/*.flac', recursive=True))
        already = next(
            (f for f in existing
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if already:
            audio_file_map[key] = already
            print(f'  Cached  : {os.path.basename(already)}')
            continue

        # Small window: 5 s before to 55 s after the recording start.
        # Stays within a single 60-second recording and avoids the API's
        # exclusive-lower-bound behaviour at exactly t0.
        q_min = t0 - datetime.timedelta(seconds=5)
        q_max = t0 + datetime.timedelta(seconds=55)
        try:
            client_extract.download_segments(
                dest_path=dest, stream=sid,
                min_date=q_min, max_date=q_max, parallel=False,
            )
        except Exception as e:
            print(f'  ERROR — stream {sid} at {t0}: {e}')
            continue

        all_downloaded = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                          glob.glob(f'{dest}/**/*.flac', recursive=True))
        matched = next(
            (f for f in all_downloaded
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if matched:
            audio_file_map[key] = matched
            print(f'  Downloaded : {os.path.basename(matched)}')
        else:
            print(f'  WARNING: no file found for stream {sid} at {t0}')

# ── Google Drive: match files by datetime in filename, within the site folder ─
elif AUDIO_SOURCE == 'google_drive':
    all_audio = (
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.wav',  recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.flac', recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.mp3',  recursive=True)
    )
    print(f'Audio files on Drive : {len(all_audio)}')

    def _file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            return datetime.datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                                     int(t[:2]), int(t[2:4]), int(t[4:6]))
        return None

    def _norm(s):
        # Normalise for comparison: strip accents (NFKD → drop combining marks →
        # ASCII), lowercase, keep only letters/digits. This makes 'POÇA' match
        # regardless of how the cedilla is encoded (NFC vs NFD), while still
        # keeping 'Ponto 1' distinct from 'Ponto 10'. Accent-stripping is the fix
        # for accented site names (POÇA, ÁGUA, SÃO, ...).
        s = unicodedata.normalize('NFKD', str(s))
        s = ''.join(c for c in s if not unicodedata.combining(c))
        s = s.encode('ascii', 'ignore').decode('ascii')
        return re.sub(r'[^a-z0-9]', '', s.lower())

    def _file_site_match(filepath, site):
        # Per-site subfolders: the site must be one of the FOLDER names below
        # DRIVE_AUDIO_DIR (the filename is ignored here).
        rel    = os.path.relpath(filepath, DRIVE_AUDIO_DIR)
        folders = [_norm(p) for p in rel.split(os.sep)[:-1]]
        return _norm(site) in folders

    _multi_site = unique_recs['Point name'].nunique() > 1
    n_unscoped = 0   # site folder not found
    n_notime   = 0   # no file at the expected time

    for _, rec in unique_recs.iterrows():
        site = str(rec['Point name'])
        t0   = rec['_rec_dt']
        key  = (site, t0)

        # Restrict to this site's subfolder so two sites recorded at the same
        # scheduled time can never be confused.
        site_files = [f for f in all_audio if _file_site_match(f, site)]

        if site_files:
            search = site_files
        elif _multi_site:
            # Fail loudly instead of guessing. With several sites, matching by
            # time alone can silently grab another site's recording (recorders
            # are usually synchronised, so every site has a file at this exact
            # time). A missing clip is noticed; a wrong clip is not.
            print(f'  ERROR: no subfolder matched site "{site}" — skipping this '
                  f'recording. Rename a subfolder under the audio path to match the '
                  f'"site" value ("{site}"), then re-run.')
            n_unscoped += 1
            continue
        else:
            # Single site: there is no other site to confuse it with.
            search = all_audio

        time_hits = sorted(
            f for f in search
            if (fdt := _file_dt(f)) and abs((fdt - t0).total_seconds()) < 5
        )
        if not time_hits:
            print(f'  WARNING: no audio file for site "{site}" at {t0} — skipping.')
            n_notime += 1
            continue

        matched = time_hits[0]
        audio_file_map[key] = matched
        if len(time_hits) > 1:
            print(f'  WARNING: {len(time_hits)} files match site "{site}" at {t0}; '
                  f'using {os.path.basename(matched)}.')
        else:
            print(f'  Matched : {site} -> {os.path.basename(matched)}')

    if n_unscoped:
        print(f'\n{n_unscoped} recording(s) skipped — site folder not found. '
              f'These detections will NOT be extracted until the folder names '
              f'match the "site" column.')
    if n_notime:
        print(f'{n_notime} recording(s) skipped — no audio file at the expected time.')

print(f'\nAudio files ready : {len(audio_file_map)} / {len(unique_recs)}')

In [195]:
#@title ✂️ Extract and save audio segments { display-mode: "form" }
#@markdown Cuts each detection out of its recording and saves it as a WAV file.
#@markdown Clips are saved into one subfolder per site, then per label, inside the output folder.
#@markdown
#@markdown Detection times are offsets from the start of the recording (0 = beginning),
#@markdown so the clip is cut at start_time..end_time within the matched file.
#@markdown
#@markdown File names follow the pattern:
#@markdown `SITE_YYYYMMDD_HHMMSS_START-ENDs_CONF_LABEL.wav`

import librosa, soundfile as sf
import numpy as np

saved   = 0
skipped = 0
errors  = 0

for _, row in df_ext.iterrows():
    if AUDIO_SOURCE == 'arbimon':
        key = (str(row['stream_id']), row['_rec_dt'])
    else:
        key = (str(row['Point name']), row['_rec_dt'])
    audio_path = audio_file_map.get(key)

    if not audio_path:
        errors += 1
        continue

    # start_time/end_time are seconds from the start of the recording (0-based).
    det_start  = float(row['start_time'])
    det_end    = float(row['end_time'])
    start_s    = max(0.0, det_start - SEGMENT_PADDING_S)
    end_s      = det_end + SEGMENT_PADDING_S
    conf       = float(row['Confidence'])

    site_safe  = str(row['Point name']).replace(' ', '_')
    label_safe = str(row['Common name']).replace(' ', '_').replace('/', '-')
    date_str   = str(row['Date']).replace('-', '')
    time_str   = str(row['Time']).replace(':', '')[:6]
    conf_str   = f'{conf:.3f}'
    # Keep the fractional detection start in the name so two detections in the
    # same second are not collapsed onto one filename (and silently skipped).
    start_tag  = f'{det_start:.1f}'
    end_tag    = f'{det_end:.1f}'
    out_name   = f'{site_safe}_{date_str}_{time_str}_{start_tag}-{end_tag}s_{conf_str}_{label_safe}.wav'

    seg_out = os.path.join(DRIVE_SEGMENTS_DIR, site_safe, label_safe)
    os.makedirs(seg_out, exist_ok=True)
    out_path = os.path.join(seg_out, out_name)

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        audio, _ = librosa.load(audio_path, sr=EXTRACT_SR, mono=True,
                                offset=start_s, duration=end_s - start_s)
    except Exception as e:
        print(f'  ERROR loading {os.path.basename(audio_path)}: {e}')
        errors += 1
        continue

    sf.write(out_path, audio, EXTRACT_SR)
    saved += 1

print(f'Segments saved     : {saved}')
if skipped:
    print(f'Already existed    : {skipped} (skipped)')
if errors:
    print(f'Skipped (no audio) : {errors}')
print(f'Output folder      : {DRIVE_SEGMENTS_DIR}')

---
## Step 6 — Review Extracted Segments <a name="step-6"></a>

Browse the audio clips extracted in Step 5 one at a time.
Each segment is shown as a spectrogram with its label and confidence score.
You can listen to it with the built-in audio player, then mark it as **true** (a genuine detection) or **false** (a false positive).

- Marking a segment moves its file into a `true/` or `false/` subfolder inside the segments folder.
- Use **← Prev** and **Next →** to browse without making a decision.
- Already-reviewed files (inside `true/` or `false/`) are excluded from the list automatically each time you run the cell.
- The **spectrogram controls** (type, frequency range, dB floor) update the view instantly without reloading the audio.

> Run Step 5 first so the segments folder is populated, then run the two cells below.

In [196]:
#@title ⚙️ Review settings { display-mode: "form" }
#@markdown Full path to the folder containing the extracted segments.
#@markdown Use the same value as **DRIVE_SEGMENTS_DIR** from Step 5.
REVIEW_DIR = "/content/drive/MyDrive/segments"  #@param {type:"string"}

#@markdown ---
#@markdown ### Spectrogram defaults
#@markdown These set the initial values shown in the reviewer.
#@markdown You can adjust all of them live inside the GUI without re-running this cell.

#@markdown **Spectrogram type** — `mel` uses a perceptual frequency scale; `fft` uses a linear Hz scale.
SPEC_TYPE = "mel"  #@param ["mel", "fft"]

#@markdown **Minimum frequency (Hz)** — lower bound of the frequency axis.
FREQ_MIN_HZ = 0  #@param {type:"integer"}

#@markdown **Maximum frequency (Hz)** — upper bound. Set to **0** to use the Nyquist frequency (half the sample rate).
FREQ_MAX_HZ = 0  #@param {type:"integer"}

#@markdown **dB floor** — lowest value shown on the colour scale (e.g. −80). Typical range: −100 to −40.
DB_MIN = -80  #@param {type:"integer"}

import os
os.makedirs(os.path.join(REVIEW_DIR, 'true'),  exist_ok=True)
os.makedirs(os.path.join(REVIEW_DIR, 'false'), exist_ok=True)

import glob as _g
_all  = _g.glob(f'{REVIEW_DIR}/**/*.wav', recursive=True)
_skip = {os.path.join(REVIEW_DIR, 'true'), os.path.join(REVIEW_DIR, 'false')}
_pending = [f for f in _all
            if not any(os.path.commonpath([f, s]) == s for s in _skip)]

print(f'Segments folder  : {REVIEW_DIR}')
print(f'Pending review   : {len(_pending)} segment(s)')
print(f'Already true     : {len(_g.glob(os.path.join(REVIEW_DIR, "true",  "*.wav")))}')
print(f'Already false    : {len(_g.glob(os.path.join(REVIEW_DIR, "false", "*.wav")))}')
print(f'Spectrogram type : {SPEC_TYPE}')
print(f'Freq range       : {FREQ_MIN_HZ} – {"Nyquist" if FREQ_MAX_HZ == 0 else str(FREQ_MAX_HZ) + " Hz"}')
print(f'dB floor         : {DB_MIN} dB')

In [197]:
#@title 🎧 Segment reviewer { display-mode: "form" }
#@markdown Run this cell to open the interactive reviewer.
#@markdown Use **True / False** to classify a segment and move to the next one.
#@markdown Use **← Prev / Next →** to browse without classifying.
#@markdown Adjust the spectrogram controls to change the view — the audio player is unaffected.
#@markdown When **False** is pressed, enter the correct label before confirming.

import os, glob, shutil, re, io, base64
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# 1x1 transparent PNG — assigning b'' does not reliably clear an Image
# widget in Colab, which can leave the previous segment's spectrogram on
# screen next to the new audio. Use this as an explicit 'blank'.
_BLANK_PNG = base64.b64decode(
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=='
)

# ── Collect unreviewed segments ───────────────────────────────────────────────
def _collect(base):
    skip = {os.path.join(base, 'true'), os.path.join(base, 'false')}
    return sorted(
        f for f in glob.glob(f'{base}/**/*.wav', recursive=True)
        if not any(os.path.commonpath([f, s]) == s for s in skip)
    )

# ── Parse label + confidence from filename ────────────────────────────────────
def _parse(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    # Confidence is the last _<number>_ token before the label; the greedy
    # prefix forces the match onto it even if other decimals appear earlier.
    m = re.search(r'.*_(\d+\.\d+)_(.+)$', stem)
    if m:
        return m.group(2).replace('_', ' '), float(m.group(1))
    return stem, None

# ── State ─────────────────────────────────────────────────────────────────────
_state = {'idx': 0, 'segs': _collect(REVIEW_DIR)}

# ── Spectrogram controls ──────────────────────────────────────────────────────
_W_ctrl = {'style': {'description_width': 'initial'}}

_dd_type = widgets.Dropdown(
    options=[('Mel', 'mel'), ('FFT (linear)', 'fft')],
    value=globals().get('SPEC_TYPE', 'mel'),
    description='Type:',
    layout=widgets.Layout(width='165px'),
    **_W_ctrl,
)
_int_fmin = widgets.BoundedIntText(
    value=globals().get('FREQ_MIN_HZ', 0), min=0, max=96000, step=100,
    description='Min Hz:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_fmax = widgets.BoundedIntText(
    value=globals().get('FREQ_MAX_HZ', 0), min=0, max=96000, step=100,
    description='Max Hz:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_db = widgets.BoundedIntText(
    value=globals().get('DB_MIN', -80), min=-120, max=-20, step=5,
    description='dB floor:',
    layout=widgets.Layout(width='160px'),
    **_W_ctrl,
)
_ctrl_row = widgets.HBox(
    [_dd_type, _int_fmin, _int_fmax, _int_db,
     widgets.Label('(Max Hz = 0 → Nyquist)', layout=widgets.Layout(margin='6px 0 0 4px'))],
    layout=widgets.Layout(justify_content='center', margin='4px 0', flex_wrap='wrap'),
)

# ── Display widgets ───────────────────────────────────────────────────────────
_spec_widget  = widgets.Image(value=b'', format='png',
                               layout=widgets.Layout(width='720px', height='auto'))
_audio_widget = widgets.HTML(value='')
_lbl_prog     = widgets.HTML(value='')
_lbl_error    = widgets.HTML(value='')

# ── Navigation row ────────────────────────────────────────────────────────────
_BW = widgets.Layout(width='120px', height='36px')
_btn_prev  = widgets.Button(description='← Prev',   layout=widgets.Layout(width='100px', height='36px'))
_btn_true  = widgets.Button(description='✔  True',  button_style='success', layout=_BW)
_btn_false = widgets.Button(description='✘  False', button_style='danger',  layout=_BW)
_btn_next  = widgets.Button(description='Next →',   layout=widgets.Layout(width='100px', height='36px'))

_nav_row = widgets.HBox(
    [_btn_prev, _btn_true, _btn_false, _btn_next],
    layout=widgets.Layout(justify_content='center', margin='8px 0'),
)

# ── False-label input row (hidden until False is pressed) ─────────────────────
_input_label       = widgets.Text(
    placeholder='Correct label  (leave blank → "unknown")',
    layout=widgets.Layout(width='340px', height='36px'),
)
_btn_confirm_false = widgets.Button(description='Confirm', button_style='warning',
                                     layout=widgets.Layout(width='100px', height='36px'))
_btn_cancel        = widgets.Button(description='Cancel',
                                     layout=widgets.Layout(width='80px',  height='36px'))

_label_row = widgets.VBox([
    widgets.HTML('<span style="font-size:12px">Enter the correct label for this segment:</span>'),
    widgets.HBox(
        [_input_label, _btn_confirm_false, _btn_cancel],
        layout=widgets.Layout(justify_content='center', margin='4px 0'),
    ),
], layout=widgets.Layout(align_items='center', margin='4px 0', display='none'))

# ── Full UI ───────────────────────────────────────────────────────────────────
_ui = widgets.VBox([
    _lbl_prog,
    _ctrl_row,
    _spec_widget,
    _audio_widget,
    _lbl_error,
    _label_row,
    _nav_row,
], layout=widgets.Layout(align_items='center'))

# ── Spectrogram rendering ─────────────────────────────────────────────────────
def _show_spec():
    segs = _state['segs']
    if not segs:
        _spec_widget.value = _BLANK_PNG
        return
    filepath = segs[_state['idx']]
    label, conf = _parse(filepath)
    conf_str  = f'{conf:.3f}' if conf is not None else '?'
    spec_type = _dd_type.value
    fmin_hz   = _int_fmin.value
    db_floor  = _int_db.value
    _lbl_error.value = ''
    try:
        y, sr   = librosa.load(filepath, sr=None, mono=True)
        fmax_hz = _int_fmax.value if _int_fmax.value > 0 else sr // 2
        fig, ax = plt.subplots(figsize=(9, 4))
        if spec_type == 'mel':
            S   = librosa.feature.melspectrogram(
                y=y, sr=sr, n_mels=128, fmin=fmin_hz, fmax=fmax_hz)
            Sd  = librosa.power_to_db(S, ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='mel', ax=ax,
                fmin=fmin_hz, fmax=fmax_hz, vmin=db_floor, vmax=0)
        else:
            D   = librosa.stft(y)
            Sd  = librosa.amplitude_to_db(np.abs(D), ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='hz', ax=ax,
                vmin=db_floor, vmax=0)
            ax.set_ylim(fmin_hz, fmax_hz)
        ax.set_title(f'{label}   |   score: {conf_str}', fontsize=13, fontweight='bold')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        plt.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        _spec_widget.value = buf.read()
    except Exception as e:
        plt.close('all')
        _spec_widget.value = _BLANK_PNG
        _lbl_error.value = f'<span style="color:red">Spectrogram error: {e}</span>'

def _show_audio():
    segs = _state['segs']
    if not segs:
        _audio_widget.value = ''
        return
    try:
        with open(segs[_state['idx']], 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        _audio_widget.value = (
            f'<audio controls style="width:720px;margin-top:4px">'
            f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
            f'</audio>'
        )
    except Exception as e:
        _audio_widget.value = f'<span style="color:red">Audio error: {e}</span>'

def _show():
    segs = _state['segs']
    n    = len(segs)
    if n == 0:
        _lbl_prog.value     = '<b>All segments have been reviewed.</b>'
        _spec_widget.value  = _BLANK_PNG
        _audio_widget.value = ''
        return
    _lbl_prog.value = (
        f'<span style="font-size:13px">Segment <b>{_state["idx"] + 1}</b> of <b>{n}</b></span>'
    )
    _show_spec()
    _show_audio()

# ── File operations ───────────────────────────────────────────────────────────
def _move(verdict, subfolder=None):
    segs = _state['segs']
    if not segs:
        return
    dest_dir = os.path.join(REVIEW_DIR, verdict, subfolder) if subfolder \
               else os.path.join(REVIEW_DIR, verdict)
    os.makedirs(dest_dir, exist_ok=True)
    src = segs.pop(_state['idx'])
    shutil.move(src, os.path.join(dest_dir, os.path.basename(src)))
    if _state['idx'] >= len(segs) and _state['idx'] > 0:
        _state['idx'] -= 1
    _show()

def _nav(delta):
    n = len(_state['segs'])
    if n == 0:
        return
    _state['idx'] = max(0, min(n - 1, _state['idx'] + delta))
    _show()

# ── Button handlers ───────────────────────────────────────────────────────────
def _show_label_input(_):
    _input_label.value        = ''
    _label_row.layout.display = ''
    _nav_row.layout.display   = 'none'

def _on_confirm_false(_):
    raw      = _input_label.value.strip()
    subfolder = raw.replace(' ', '_').replace('/', '-') if raw else 'unknown'
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''
    _move('false', subfolder)

def _on_cancel(_):
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''

_btn_true.on_click(lambda _: _move('true'))
_btn_false.on_click(_show_label_input)
_btn_confirm_false.on_click(_on_confirm_false)
_btn_cancel.on_click(_on_cancel)
_btn_prev.on_click(lambda _: _nav(-1))
_btn_next.on_click(lambda _: _nav(+1))

# ── Spec-change observers ─────────────────────────────────────────────────────
def _on_spec_change(change):
    if _state['segs']:
        _show_spec()

for _ctrl in (_dd_type, _int_fmin, _int_fmax, _int_db):
    _ctrl.observe(_on_spec_change, names='value')

# ── Launch ────────────────────────────────────────────────────────────────────
_show()
display(_ui)